In [1]:
# 1. Install necessary libraries for Japanese support and plotting
!pip install japanize-matplotlib -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 73.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import pandas as pd
import glob
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
import os

import re

import csv
import unicodedata
import matplotlib
import openpyxl

import sys

import matplotlib.font_manager as fm
from openpyxl import load_workbook


In [3]:
"""
退院時転帰「死亡」合計 集計スクリプト - ロバスト走査版
=======================================================
【走査ロジック】
1. keyシートから調査年度を自動取得（例: 20190630 -> 2019）
2. 全シートを走査し「退院」「転帰」を含むシートを特定
3. 特定シート内でA列=「死亡」の行を探し、B列（合計）を取得
4. 複数の退院転帰シートがあっても合計列は同値 -> 最初の有効値を採用
5. 病院コードで名寄せ（病院名の表記揺れを吸収）

【フォルダ構成】
/kaggle/input/datasets/kenkuroiwa/630mental
  2019/2019/  <- .xlsxが入っている
  2020/2020/
  2021/2021/
  2022/2022/
  2023/2023/
  2024/2024/

【出力】
  集計結果_全年度.csv   ... 病院×年度のレコード形式
  集計結果_ピボット.csv ... 病院名寄せ×年度のクロス集計
  集計エラーログ.csv    ... 処理できなかったファイル一覧
"""


'\n退院時転帰「死亡」合計 集計スクリプト - ロバスト走査版\n=======================================================\n【走査ロジック】\n1. keyシートから調査年度を自動取得（例: 20190630 -> 2019）\n2. 全シートを走査し「退院」「転帰」を含むシートを特定\n3. 特定シート内でA列=「死亡」の行を探し、B列（合計）を取得\n4. 複数の退院転帰シートがあっても合計列は同値 -> 最初の有効値を採用\n5. 病院コードで名寄せ（病院名の表記揺れを吸収）\n\n【フォルダ構成】\n/kaggle/input/datasets/kenkuroiwa/630mental\n  2019/2019/  <- .xlsxが入っている\n  2020/2020/\n  2021/2021/\n  2022/2022/\n  2023/2023/\n  2024/2024/\n\n【出力】\n  集計結果_全年度.csv   ... 病院×年度のレコード形式\n  集計結果_ピボット.csv ... 病院名寄せ×年度のクロス集計\n  集計エラーログ.csv    ... 処理できなかったファイル一覧\n'

In [4]:

# ============================================================
# 設定
# ============================================================
BASE_DIR = r"/kaggle/input/datasets/kenkuroiwa/630mental"

YEAR_FOLDERS = {
    "2019": os.path.join(BASE_DIR, "2019", "2019"),
    "2020": os.path.join(BASE_DIR, "2020", "2020"),
    "2021": os.path.join(BASE_DIR, "2021", "2021"),
    "2022": os.path.join(BASE_DIR, "2022", "2022"),
    "2023": os.path.join(BASE_DIR, "2023", "2023"),
    "2024": os.path.join(BASE_DIR, "2024", "2024"),
}

OUTPUT_RECORDS = os.path.join("/kaggle/working/集計結果_全年度.csv")
OUTPUT_PIVOT   = os.path.join("/kaggle/working/集計結果_ピボット.csv")
OUTPUT_ERRORS  = os.path.join("/kaggle/working/集計エラーログ.csv")
# ============================================================



In [5]:
def get_year_from_key_sheet(wb):
    """keyシートから年度を取得（例: 20190630 -> "2019"）"""
    for ws in wb.worksheets:
        if ws.title.strip().lower() == "key":
            for row in ws.iter_rows(values_only=True, max_row=5):
                for cell in row:
                    if cell is None:
                        continue
                    try:
                        s = str(int(cell)) if isinstance(cell, (int, float)) else str(cell)
                    except Exception:
                        s = str(cell)
                    m = re.match(r"(\d{4})\d{4}", s)
                    if m:
                        return m.group(1)
    return None



In [6]:
def find_hospital_info(wb):
    """
    病院名・病院コードを取得。
    「医療機関名」ヘッダーの次行同列を検索。
    「厚生局届出の医療機関番号」も同様に取得。
    """
    hospital_name = None
    hospital_code = None
    for ws in wb.worksheets:
        rows = list(ws.iter_rows(values_only=True, max_row=10))
        name_col = None
        code_col = None
        for r, row in enumerate(rows):
            for c, cell in enumerate(row):
                if cell and "医療機関名" in str(cell):
                    name_col = c
                if cell and "医療機関番号" in str(cell):
                    code_col = c
            # ヘッダー行の次行から値を取得
            if name_col is not None or code_col is not None:
                next_r = r + 1
                if next_r < len(rows):
                    if name_col is not None and name_col < len(rows[next_r]):
                        v = rows[next_r][name_col]
                        if v and str(v).strip():
                            hospital_name = str(v).strip()
                    if code_col is not None and code_col < len(rows[next_r]):
                        v = rows[next_r][code_col]
                        if v:
                            hospital_code = str(v).strip()
                if hospital_name:
                    return hospital_name, hospital_code
    return hospital_name, hospital_code



In [7]:
def find_death_total(wb):
    """
    退院転帰シートを走査して死亡合計を取得。
    - 「退院」「転帰」を含むシートを対象
    - A列=「死亡」の行のB列（合計）を取得
    - 複数シートで同値のため最初の有効値を返す
    - 提出調査票49など死亡行がないシートは自動スキップ
    返り値: (death_total, tenki_sheet_count, detail_log)
    """
    death_total = None
    tenki_count = 0
    detail = []

    for idx, ws in enumerate(wb.worksheets):
        # 先頭20行だけ読んで退院転帰シートか判定
        header_rows = list(ws.iter_rows(values_only=True, max_row=20))
        is_tenki = any(
            any(c and "退院" in str(c) and "転帰" in str(c) for c in row)
            for row in header_rows
        )
        if not is_tenki:
            continue

        tenki_count += 1
        # 死亡行を走査（最大30行）
        all_rows = list(ws.iter_rows(values_only=True, max_row=30))
        for r, row in enumerate(all_rows):
            a = row[0] if row else None
            if a and str(a).strip() == "死亡":
                raw_val = row[1] if len(row) > 1 else None
                try:
                    val = int(raw_val) if raw_val is not None else 0
                except (ValueError, TypeError):
                    val = 0
                detail.append(f"[{idx}]{ws.title} 行{r+1} 死亡={val}")
                if death_total is None:
                    death_total = val  # 最初の有効値を採用
                break
        else:
            detail.append(f"[{idx}]{ws.title} 死亡行なし（スキップ）")

    return death_total, tenki_count, "; ".join(detail)



In [8]:
def process_file(filepath, folder_year):
    """
    1ファイルを処理してレコードdictを返す。
    エラー時は (None, エラー文字列) を返す。
    """
    filename = os.path.basename(filepath)
    try:
        wb = load_workbook(filepath, read_only=True, data_only=True)
    except Exception as e:
        return None, f"ファイル読み込みエラー: {e}"

    # 年度取得（keyシート優先、なければフォルダ年度）
    key_year = get_year_from_key_sheet(wb)
    year = key_year if key_year else folder_year

    # 病院名・コード取得
    hospital_name, hospital_code = find_hospital_info(wb)
    if not hospital_name:
        parts = filename.replace(".xlsx", "").split("_")
        hospital_name = parts[-1] if len(parts) >= 3 else filename.replace(".xlsx", "")
    if not hospital_code:
        parts = filename.replace(".xlsx", "").split("_")
        hospital_code = parts[1] if len(parts) >= 2 else ""

    # 死亡合計取得
    death_total, tenki_count, detail_log = find_death_total(wb)
    wb.close()

    if death_total is None:
        return None, f"死亡行が見つかりません（転帰シート数={tenki_count}）| {detail_log}"

    return {
        "年度":           year,
        "病院コード":     hospital_code,
        "病院名":         hospital_name,
        "死亡合計":       death_total,
        "転帰シート数":   tenki_count,
        "ファイル名":     filename,
    }, None


In [9]:
# ============================================================
# メイン処理
# ============================================================
results = []
errors  = []

for folder_year, folder_path in YEAR_FOLDERS.items():
    if not os.path.exists(folder_path):
        print(f"[スキップ] フォルダなし: {folder_path}")
        continue

    xlsx_files = sorted(glob.glob(os.path.join(folder_path, "*.xlsx")))
    print(f"\n{folder_year}年度: {len(xlsx_files)}ファイル処理中...")

    for i, filepath in enumerate(xlsx_files):
        filename = os.path.basename(filepath)
        record, err = process_file(filepath, folder_year)
        if record:
            results.append(record)
        else:
            errors.append({
                "ファイル": filename,
                "フォルダ年度": folder_year,
                "エラー": err,
            })
        # 進捗表示
        if (i + 1) % 50 == 0 or (i + 1) == len(xlsx_files):
            ok = len([r for r in results if r["年度"] == folder_year])
            print(f"  {i+1}/{len(xlsx_files)} 処理済み（成功: {ok}件）")

print(f"\n処理完了: 成功 {len(results)}件 / エラー {len(errors)}件")




2019年度: 99ファイル処理中...
  50/99 処理済み（成功: 50件）
  99/99 処理済み（成功: 99件）
[スキップ] フォルダなし: /kaggle/input/datasets/kenkuroiwa/630mental/2020/2020

2021年度: 96ファイル処理中...
  50/96 処理済み（成功: 50件）
  96/96 処理済み（成功: 96件）

2022年度: 98ファイル処理中...
  50/98 処理済み（成功: 50件）
  98/98 処理済み（成功: 98件）

2023年度: 99ファイル処理中...
  50/99 処理済み（成功: 50件）
  99/99 処理済み（成功: 99件）

2024年度: 96ファイル処理中...
  50/96 処理済み（成功: 50件）
  96/96 処理済み（成功: 96件）

処理完了: 成功 488件 / エラー 0件


In [10]:
# ============================================================
# 出力
# ============================================================
df = pd.DataFrame(results)

if df.empty:
    print("[エラー] データが取得できませんでした。フォルダパスを確認してください。")
else:
    # --- レコード形式 ---
    df_out = df[["年度","病院コード","病院名","死亡合計","転帰シート数","ファイル名"]]
    df_out = df_out.sort_values(["年度","死亡合計"], ascending=[True, False])
    df_out.to_csv(OUTPUT_RECORDS, index=False, encoding="utf-8-sig")
    print(f"\n[出力1] レコード形式: {OUTPUT_RECORDS}")

    # --- ピボット（病院コードで名寄せ） ---
    # 病院コードが同じなら同一病院とみなし、代表病院名（最新）を使う
    name_map = df.sort_values("年度").groupby("病院コード")["病院名"].last()
    pivot = df.pivot_table(
        index="病院コード",
        columns="年度",
        values="死亡合計",
        aggfunc="sum",
        fill_value=0
    )
    pivot.insert(0, "病院名", pivot.index.map(name_map))
    pivot["全年度合計"] = pivot.drop(columns="病院名").sum(axis=1)
    pivot = pivot.sort_values("全年度合計", ascending=False)
    pivot.to_csv(OUTPUT_PIVOT, encoding="utf-8-sig")
    print(f"[出力2] ピボット（名寄せ）: {OUTPUT_PIVOT}")

    # --- 年度別サマリー ---
    print(f"\n{'='*55}")
    print("  年度別サマリー")
    print(f"{'='*55}")
    summary = df.groupby("年度")["死亡合計"].agg(
        病院数="count",
        合計="sum",
        死亡あり病院数=lambda x: (x > 0).sum(),
        平均="mean",
        最大="max"
    ).round(2)
    print(summary.to_string())

    # --- 全年度 上位15病院 ---
    print(f"\n{'='*55}")
    print("  全年度合計 上位15病院（病院コードで名寄せ）")
    print(f"{'='*55}")
    top = pivot.head(15)[["病院名","全年度合計"] + [c for c in pivot.columns if str(c).isdigit()]]
    print(top.to_string())

# --- エラーログ ---
if errors:
    pd.DataFrame(errors).to_csv(OUTPUT_ERRORS, index=False, encoding="utf-8-sig")
    print(f"\n[注意] エラー {len(errors)}件 -> {OUTPUT_ERRORS}")
else:
    print("\n[OK] エラーなし")


[出力1] レコード形式: /kaggle/working/集計結果_全年度.csv
[出力2] ピボット（名寄せ）: /kaggle/working/集計結果_ピボット.csv

  年度別サマリー
      病院数   合計  死亡あり病院数    平均  最大
年度                               
2019   99  122       46  1.23  11
2021   96  138       47  1.44   9
2022   98  134       49  1.37   8
2023   99  141       52  1.42   9
2024   96  156       51  1.62  12

  全年度合計 上位15病院（病院コードで名寄せ）
年度                         病院名  全年度合計  2019  2021  2022  2023  2024
病院コード                                                              
3270022  医療法人財団　明理会　鶴川サナトリウム病院     36     5     6     7     9     9
2870848       医療法人社団純正会　青梅東部病院     31     4     8     2     5    12
3613791           公益財団法人　井之頭病院     26     8     3     8     3     4
2919728       医療法人社団小松会　聖パウロ病院     26    11     3     4     4     4
2971232         医療法人社団孝山会　滝山病院     23     7     9     3     2     2
5119615                  稲城台病院     22     2     7     7     4     2
3270808            こころのホスピタル町田     20     2     7     5     3     3
2015154              

In [11]:

# Search for all .xlsx files recursively
root_dir = "/kaggle/input/datasets/kenkuroiwa/630mental"
xlsx_files = glob.glob(os.path.join(root_dir, "**", "*.xlsx"), recursive=True)
xlsx_files = sorted(list(set(xlsx_files)))

In [12]:
results = []

def clean_val(v):
    if v is None: return ""
    return str(v).strip()


In [13]:


def get_hospital_name(file_path):
    base = os.path.basename(file_path)
    name_no_ext = os.path.splitext(base)[0]
    parts = name_no_ext.split("_")
    if len(parts) >= 3:
        return parts[2]
    elif len(parts) == 2:
        return parts[1]
    return name_no_ext

In [14]:



def get_year_from_path(file_path):
    parts = file_path.split(os.sep)
    for p in parts:
        if p.isdigit() and len(p) == 4:
            return int(p)
    return None

print(f"Found {len(xlsx_files)} Excel files. Starting processing...", flush=True)


Found 502 Excel files. Starting processing...


In [15]:


for i, file_path in enumerate(xlsx_files):
    
    hospital_name = get_hospital_name(file_path)
    year = get_year_from_path(file_path)
    
    if year is None: continue

    try:
        xl = pd.ExcelFile(file_path, engine='openpyxl')
        file_death_total = 0
        for sheet_name in xl.sheet_names:
            df = pd.read_excel(xl, sheet_name=sheet_name, header=None)
            
            r_header = -1
            c_label = -1
            for r_idx, row in df.iterrows():
                for c_idx, value in enumerate(row):
                    if clean_val(value) == "退院後転帰":
                        r_header = r_idx
                        c_label = c_idx
                        break
                if r_header != -1:
                    break
            
            if r_header != -1:
                c_total = -1
                for c_idx in range(c_label + 1, len(df.columns)):
                    val = clean_val(df.iloc[r_header, c_idx])
                    if "合計" in val or val == "計":
                        c_total = c_idx
                        break
                
                if c_total == -1 and r_header + 1 < len(df):
                    for c_idx in range(c_label + 1, len(df.columns)):
                        val = clean_val(df.iloc[r_header + 1, c_idx])
                        if "合計" in val or val == "計":
                            c_total = c_idx
                            break

                if c_total != -1:
                    for r_idx in range(r_header + 1, len(df)):
                        label_val = clean_val(df.iloc[r_idx, c_label])
                        if "死亡" in label_val:
                            death_val = df.iloc[r_idx, c_total]
                            numeric_val = 0
                            if pd.notnull(death_val) and isinstance(death_val, (int, float)):
                                numeric_val = death_val
                            elif isinstance(death_val, str):
                                try:
                                    numeric_val = float(death_val)
                                except ValueError:
                                    pass
                            file_death_total += numeric_val
                            break
        
        if file_death_total > 0:
            results.append({
                "Hospital": hospital_name,
                "Year": year,
                "DeathCount": file_death_total
            })
            # print(f"[{i+1}/{len(xlsx_files)}] {hospital_name} ({year}): {file_death_total}", flush=True)

    except Exception as e:
        pass

if not results:
    print("No data found.", flush=True)
    sys.exit()

df_raw = pd.DataFrame(results)
df_agg = df_raw.groupby(["Hospital", "Year"])["DeathCount"].sum().reset_index()


In [16]:

# Save aggregated CSV with utf-8-sig to handle special characters and be Excel-friendly
df_agg.to_csv("death_counts_aggregated.csv", index=False, encoding="utf-8-sig")
print(f"Aggregated results saved to 'death_counts_aggregated.csv'", flush=True)

# Prepare data for plotting
pivot_df = df_agg.pivot(index="Year", columns="Hospital", values="DeathCount").fillna(0)

# Sort hospitals by total deaths across all years to pick top ones for plotting
top_hospitals = pivot_df.sum().sort_values(ascending=False).head(10).index
plot_df = pivot_df[top_hospitals]

# Plotting
plt.figure(figsize=(12, 6))

for column in plot_df.columns:
    plt.plot(plot_df.index, plot_df[column], marker='o', label=column)

plt.title("病院別 死亡者数の推移 (上位10病院)")
plt.xlabel("年")
plt.ylabel("死亡者数")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()
plt.savefig("death_trends.png")
print(f"Time-series graph saved to 'death_trends.png'", flush=True)

total_by_year = df_agg.groupby("Year")["DeathCount"].sum()
print("\nTotal Deaths by Year:", flush=True)
print(total_by_year, flush=True)


Aggregated results saved to 'death_counts_aggregated.csv'
Time-series graph saved to 'death_trends.png'

Total Deaths by Year:
Year
2019    508
2021    552
2022    536
2023    564
2024    624
Name: DeathCount, dtype: int64
